In [1]:
from google.colab import files
uploaded = files.upload()

Saving enhanced_realistic_dataset.csv to enhanced_realistic_dataset.csv


In [2]:
import pandas as pd

df = pd.read_csv("enhanced_realistic_dataset.csv")
df.head()
import pandas as pd

df = pd.read_csv("enhanced_realistic_dataset.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24000 entries, 0 to 23999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   product_id        24000 non-null  int64  
 1   brand             24000 non-null  object 
 2   product_name      24000 non-null  object 
 3   category          24000 non-null  object 
 4   price             24000 non-null  int64  
 5   rating            24000 non-null  int64  
 6   review_count      22793 non-null  float64
 7   sales_count       24000 non-null  int64  
 8   review_text       20315 non-null  object 
 9   review_sentiment  22796 non-null  object 
 10  description       22046 non-null  object 
 11  image_url         24000 non-null  object 
 12  in_stock          24000 non-null  bool   
dtypes: bool(1), float64(1), int64(4), object(7)
memory usage: 2.2+ MB


In [4]:
df.isnull().sum()

,0
product_id,0
brand,0
product_name,0
category,0
price,0
rating,0
review_count,1207
sales_count,0
review_text,3685
review_sentiment,1204


In [5]:
df.describe()

,product_id,price,rating,review_count,sales_count
count,24000.000000,24000.000000,24000.000000,22793.000000,24000.000000
mean,111999.500000,309.248625,4.007333,4536.288203,26528.548875
std,6928.347566,143.018878,1.216110,5083.150350,32352.865913
min,100000.000000,70.000000,1.000000,52.000000,248.000000
25%,105999.750000,194.000000,3.000000,1640.000000,8290.750000
50%,111999.500000,292.000000,4.000000,3207.000000,17536.000000
75%,117999.250000,400.000000,5.000000,5114.000000,30336.250000
max,123999.000000,750.000000,5.000000,38894.000000,335039.000000


In [6]:
# الأعمدة المهمة: product_name, category, description
# أي صف ناقصهم هتمسح لأنه مش هيخدم الـ chatbot
critical_cols = ['product_name', 'category', 'description']
initial_rows = df.shape[0]

df_clean = df.dropna(subset=critical_cols)
df_clean.reset_index(drop=True, inplace=True)

print(f"عدد الصفوف المحذوفة بسبب missing في الأعمدة المهمة: {initial_rows - df_clean.shape[0]}")



# review_text → placeholder لو فاضي
df_clean['review_text'] = df_clean['review_text'].fillna("No review provided")

# review_count → 0 لو فاضي
df_clean['review_count'] = df_clean['review_count'].fillna(0)

# in_stock → True لو فاضي
df_clean['in_stock'] = df_clean['in_stock'].fillna(True)

#  تنظيف النصوص
text_cols = ['brand', 'product_name', 'category', 'review_text', 'description']
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.lower().str.strip()

# أي صف ناقصهم هتمسح لأنه مش هيخدم الـ chatbot
critical_cols = ['product_name', 'category', 'description']
initial_rows = df.shape[0]

df_clean = df.dropna(subset=critical_cols)
df_clean.reset_index(drop=True, inplace=True)

print(f"عدد الصفوف المحذوفة بسبب missing في الأعمدة المهمة: {initial_rows - df_clean.shape[0]}")



#  إزالة التكرارات
# نتأكد إن كل product_id فريد
df_clean = df_clean.drop_duplicates(subset=['product_id'])

#  الفحص النهائي
print("الشكل بعد التنظيف:", df_clean.shape)
print("\nMissing values لكل عمود بعد التنظيف:\n", df_clean.isnull().sum())


عدد الصفوف المحذوفة بسبب missing في الأعمدة المهمة: 1954
عدد الصفوف المحذوفة بسبب missing في الأعمدة المهمة: 1954
الشكل بعد التنظيف: (22046, 13)

Missing values لكل عمود بعد التنظيف:
 product_id             0
brand                  0
product_name           0
category               0
price                  0
rating                 0
review_count        1089
sales_count            0
review_text         3352
review_sentiment    1113
description            0
image_url              0
in_stock               0
dtype: int64


/tmp/ipython-input-3563523646.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['review_text'] = df_clean['review_text'].fillna("No review provided")
/tmp/ipython-input-3563523646.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['review_count'] = df_clean['review_count'].fillna(0)
/tmp/ipython-input-3563523646.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 

In [7]:
import re

def clean_text(text):
    # إزالة الإيموجيز
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    # إزالة الرموز الغريبة
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # إزالة تكرار المسافات
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# تطبيق الدالة على أعمدة النصوص
text_cols = ['brand', 'product_name', 'category', 'description', 'review_text']

for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).apply(clean_text)


In [8]:
#  تطبيق الـ preprocessing على الأعمدة النصية
# =================================================

# الأعمدة النصية اللي هنشتغل عليها (لازم تكون موجودة من الخطوة اللي قبل)
text_cols = ['brand', 'product_name', 'category', 'review_text', 'description']

# تطبيق تنظيف النص على كل عمود
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).apply(clean_text)

print(" تم تطبيق الـ text preprocessing بنجاح!")


 تم تطبيق الـ text preprocessing بنجاح!


In [9]:
#  إنشاء عمود clean_doc لتجهيز البيانات للـ Search والـ Embeddings
# =====================================================================

df_clean['clean_doc'] = (
    df_clean['product_name'] + " " +
    df_clean['category'] + " " +
    df_clean['description'] + " " +
    df_clean['review_text']
)

print(" تم إنشاء عمود clean_doc بنجاح!")


 تم إنشاء عمود clean_doc بنجاح!


In [10]:
# Tokenization + Preprocessing إضافي قبل الـ Embeddings
# ===========================================================

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Stopwords إنجليزي
stop_words = set(stopwords.words("english"))

def tokenize_text(text):
    # تحويل النص لقائمة كلمات
    tokens = word_tokenize(text)

    # إزالة علامات الترقيم والكلمات الصغيرة جدًا
    tokens = [w for w in tokens if w.isalpha() and len(w) > 2]

    # إزالة stopwords
    tokens = [w for w in tokens if w not in stop_words]

    return tokens

# تطبيق التوكنز على clean_doc
df_clean["tokens"] = df_clean["clean_doc"].apply(tokenize_text)

print(" تم إنشاء عمود tokens بنجاح!")
df_clean.head()


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


 تم إنشاء عمود tokens بنجاح!


,product_id,brand,product_name,category,price,rating,review_count,sales_count,review_text,review_sentiment,description,image_url,in_stock,clean_doc,tokens
0,100000,Michael Kors,Leather Belt,Accessories,230,1,2283.0,13981,Faded significantly after the first wash despi...,NaN,This leather belt combines functionality with ...,https://images.store.com/michael kors/leather-...,True,Leather Belt Accessories This leather belt com...,"[Leather, Belt, Accessories, This, leather, be..."
1,100001,The White Company,Throw Pillow,Home Living,211,4,2095.0,14781,I m usually between sizes and ordered my usual...,Positive,Soft throw pillow that adds both comfort and s...,https://images.store.com/the white company/thr...,True,Throw Pillow Home Living Soft throw pillow tha...,"[Throw, Pillow, Home, Living, Soft, throw, pil..."
2,100002,Concrete,Linen Pants,Clothing,179,5,4175.0,18382,nan,Positive,Premium linen pants that delivers on both form...,https://images.store.com/concrete/linen-pants-...,True,Linen Pants Clothing Premium linen pants that ...,"[Linen, Pants, Clothing, Premium, linen, pants..."
3,100004,MAC,Matte Lipstick,Beauty Skincare,120,3,3277.0,14556,It s exactly what I expected not better not wo...,Neutral,Long lasting matte lipstick that doesn t dry o...,https://images.store.com/mac/matte-lipstick-10...,True,Matte Lipstick Beauty Skincare Long lasting ma...,"[Matte, Lipstick, Beauty, Skincare, Long, last..."
4,100005,Puma,Silk Blouse,Clothing,459,1,750.0,5485,Extremely uncomfortable to wear The fabric is ...,Negative,Elegant silk blouse that drapes beautifully Th...,https://images.store.com/puma/silk-blouse-1000...,True,Silk Blouse Clothing Elegant silk blouse that ...,"[Silk, Blouse, Clothing, Elegant, silk, blouse..."


In [11]:
#  إنشاء عمود clean_doc لتجهيز البيانات للـ Search والـ Embeddings
# =====================================================================

df_clean['clean_doc'] = (
    df_clean['product_name'] + " " +
    df_clean['category'] + " " +
    df_clean['description'] + " " +
    df_clean['review_text']
)

print(" تم إنشاء عمود clean_doc بنجاح!")

 تم إنشاء عمود clean_doc بنجاح!


In [12]:
from sentence_transformers import SentenceTransformer

# تحميل موديل MiniLM
model = SentenceTransformer('all-MiniLM-L6-v2')

print(" تم تحميل موديل Transformer ")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 تم تحميل موديل Transformer 


In [13]:
# الموديل بياخد نصوص جاهزة
documents = df_clean["clean_doc"].tolist()

# إنشاء الامبيدينجز
embeddings = model.encode(documents, batch_size=64, show_progress_bar=True)

print(" تم إنشاء الـ embeddings ")

# حفظ الـ embeddings داخل DataFrame
df_clean['clean_doc_embedding'] = list(embeddings)

Batches:   0%|          | 0/345 [00:00<?, ?it/s]

 تم إنشاء الـ embeddings 


In [14]:
#   Encoding للـ categorical features

import numpy as np
from sklearn.preprocessing import LabelEncoder

for col in ['brand', 'category']:
    le = LabelEncoder()
    df_clean[f'{col}_encoded'] = le.fit_transform(df_clean[col])


#  تجهيز الـ Feature Matrix

# تحويل embeddings لمصفوفة
X_embeddings = np.vstack(df_clean['clean_doc_embedding'].values)

# Features رقمية إضافية
X_numeric = df_clean[['price', 'sales_count', 'review_count']].values

# دمج كل الـ features في مصفوفة واحدة
X = np.hstack([X_embeddings, X_numeric])

print("Shape of X:", X.shape)

Shape of X: (22046, 387)


In [15]:
def rating_to_class(r):
    if r <= 2:
        return 0   # low
    elif r == 3:
        return 1   # medium
    else:
        return 2   # high

y = df_clean['rating'].apply(rating_to_class).values

import numpy as np
print("Distribution:", np.bincount(y))


Distribution: [ 3042  3056 15948]


In [16]:
import numpy as np

print("NaNs in embeddings:", np.isnan(X_embeddings).sum())
print("NaNs in numeric:", np.isnan(X_numeric).sum())
print("NaNs in X:", np.isnan(X).sum())


NaNs in embeddings: 0
NaNs in numeric: 1089
NaNs in X: 1089


In [17]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")

X_numeric = num_imputer.fit_transform(
    df_clean[['price', 'sales_count', 'review_count']]
)

print("NaNs in numeric بعد impute:", np.isnan(X_numeric).sum())


NaNs in numeric بعد impute: 0


In [18]:
X = np.hstack([X_embeddings, X_numeric])

print("NaNs in X بعد الدمج:", np.isnan(X).sum())
print("Final Shape:", X.shape)


NaNs in X بعد الدمج: 0
Final Shape: (22046, 387)


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np

#  Standardize Features (مهم جدًا للـ SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # X = embeddings + numeric features

#  Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y  # يحافظ على نسبة الفئات في Train/Test
)

#  Train SVM (linear kernel سريع للـ embeddings)
svm_model = SVC(
    kernel='linear',
    random_state=42,
    class_weight='balanced',  # مهم مع الداتا الغير متوازنة
    probability=True
)

# تدريب الموديل (خفيف أسرع من batch apply)
svm_model.fit(X_train, y_train)

#  Evaluation
y_pred = svm_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f} %")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

#  Optional: تعيين imdb_score تقديري لكل rating
# (لو حابة تحولي التصنيف الرقمي لقيم افتراضية)
rating_to_imdb = {0: 4.5, 1: 6.5, 2: 8.5}
df_clean['imdb_score'] = df_clean['rating'].apply(rating_to_imdb.get)


Accuracy: 89.93 %

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.93      0.83       609
           1       0.72      0.92      0.81       611
           2       0.99      0.89      0.94      3190

    accuracy                           0.90      4410
   macro avg       0.82      0.91      0.86      4410
weighted avg       0.92      0.90      0.90      4410



In [20]:
from sklearn.metrics import accuracy_score, classification_report
# Predictions على Training set
y_train_pred = svm_model.predict(X_train)

# Training Accuracy
train_acc = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_acc*100:.2f} %")

# Optional: Classification Report على Training set
print("\nTraining Classification Report:\n", classification_report(y_train, y_train_pred))
# احتمالات كل فئة لكل منتج
probs = svm_model.predict_proba(X_scaled)  # شكلها: (n_products, 3)
df_clean['high_rating_prob'] = probs[:,2]  # column index 2 = High rating


Training Accuracy: 90.92 %

Training Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.94      0.84      2433
           1       0.75      0.92      0.82      2445
           2       0.99      0.90      0.94     12758

    accuracy                           0.91     17636
   macro avg       0.83      0.92      0.87     17636
weighted avg       0.92      0.91      0.91     17636



In [21]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# ==================================================
#  تحضير Cosine Similarity بين كل المنتجات
# ==================================================
product_embeddings = np.vstack(df_clean['clean_doc_embedding'].values)
cosine_sim = cosine_similarity(product_embeddings, product_embeddings)

# ==================================================
#  إنشاء Series لتسهيل البحث عن index لكل منتج
# ==================================================
indices = pd.Series(df_clean.index, index=df_clean['product_name']).drop_duplicates()

# ==================================================
#  دالة Recommendation مصححة
# ==================================================
def recommend_products(product_name, num_recommendations=5):
    if product_name not in indices:
        return f"Product '{product_name}' not found."

    idx = indices[product_name]  # فهرس المنتج
    # نجيب التشابه مع باقي المنتجات
    sim_scores = cosine_sim[idx]  # array 1D جاهز للاستخدام
    # ترتيب الفهارس من الأعلى للأدنى
    sim_indices = np.argsort(sim_scores)[::-1]
    # إزالة المنتج نفسه من النتائج
    sim_indices = sim_indices[sim_indices != idx]

    # نجيب المنتجات المتشابهة
    recommended = df_clean.iloc[sim_indices].copy()

    # فقط أول N منتجات
    recommended = recommended.head(num_recommendations)
    sim_scores_subset = sim_scores[sim_indices][:num_recommendations]

    # ==================================================
    #  دمج High rating probability + sales_count + similarity score
    # ==================================================
    recommended['similarity_score'] = sim_scores_subset
    recommended['hybrid_score'] = (
        0.6 * recommended['high_rating_prob'] +
        0.3 * (recommended['sales_count'] / recommended['sales_count'].max()) +
        0.1 * recommended['similarity_score']
    )

    # ترتيب المنتجات حسب hybrid_score
    recommended = recommended.sort_values(by='hybrid_score', ascending=False)

    # إرجاع أعلى N منتجات
    return recommended[['product_name', 'category', 'price', 'high_rating_prob', 'sales_count', 'hybrid_score']]

In [58]:
# ==================================================
#  Hybrid Recommendation System for Products (FINAL)
# ==================================================

import re
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from sentence_transformers import SentenceTransformer



#  Zero-Shot NLP for Category Detection
nlp = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
categories = df_clean["category"].dropna().unique().tolist()

def interpret_user_request(user_input, threshold=0.4):
    result = nlp(user_input, categories)
    return [
        label
        for label, score in zip(result["labels"], result["scores"])
        if score >= threshold
    ]

#  Brand & Price Extraction (NLP-light)
brands_list = df_clean["brand"].dropna().unique().tolist()

def extract_brand(user_input):
    for brand in brands_list:
        if brand.lower() in user_input.lower():
            return brand
    return None

def extract_price_range(user_input):
    numbers = [int(n) for n in re.findall(r"\d+", user_input)]
    if len(numbers) == 1:
        return (0, numbers[0])
    elif len(numbers) >= 2:
        return (min(numbers), max(numbers))
    return None


# ============================
#  Check if user input is meaningful
# ============================
def check_user_input(user_input):
    if not user_input.strip():
        return False, "Your input is empty. Please describe what you need."

    # لو الإدخال بالعربي
    if re.search(r'[\u0600-\u06FF]', user_input):
        return False, "Please write your request in English so I can understand it."

    words = [w for w in user_input.split() if len(w) > 1]
    if len(words) < 2:
        return False, "Your input is too short or unclear. Please provide more details."

    return True, ""

#  Prepare Embeddings & Similarity
product_embeddings = np.vstack(df_clean["clean_doc_embedding"].values)
cosine_sim = cosine_similarity(product_embeddings, product_embeddings)

#  Content-Based Recommendation (Stable)
indices = pd.Series(df_clean.index, index=df_clean["product_name"]).drop_duplicates()

def content_based_recommendation(product_name, top_n=5):
    if product_name not in indices:
        return pd.DataFrame()

    idx = indices[product_name]
    sim_scores = cosine_sim[idx]
    sim_indices = np.argsort(sim_scores)[::-1]
    sim_indices = sim_indices[sim_indices != idx]

    recs = df_clean.iloc[sim_indices].head(top_n).copy()
    recs["similarity_score"] = sim_scores[sim_indices][:top_n]
    return recs

#  Hybrid Recommendation (MAIN ENGINE)
def hybrid_recommendation(user_input, top_n=5):

    # 1️⃣ فهم نية اليوزر
    detected_categories = interpret_user_request(user_input)
    extracted_brand = extract_brand(user_input)
    extracted_price = extract_price_range(user_input)

    # 2️⃣ Semantic similarity (user query vs products)
    query_embedding = model.encode([user_input])[0]
    query_sim = cosine_similarity(
        [query_embedding],
        product_embeddings
    ).flatten()

    semantic_df = df_clean.copy()
    semantic_df["query_similarity"] = query_sim
    semantic_df = semantic_df.sort_values("query_similarity", ascending=False)

    candidates = semantic_df.head(200).copy()

    # 3️⃣ Category (SAFE)
    if detected_categories:
        filtered = candidates[
            candidates["category"].str.contains(
                "|".join(detected_categories),
                case=False,
                na=False
            )
        ]
        if not filtered.empty:
            candidates = filtered

    # 4️⃣ Brand  (SAFE)
    if extracted_brand:
        filtered = candidates[
            candidates["brand"].str.contains(
                extracted_brand,
                case=False,
                na=False
            )
        ]
        if not filtered.empty:
            candidates = filtered

    # 5️⃣ Price  (SAFE)
    if extracted_price:
        min_p, max_p = extracted_price
        filtered = candidates[
            (candidates["price"] >= min_p) &
            (candidates["price"] <= max_p)
        ]
        if not filtered.empty:
            candidates = filtered


    #6️⃣  by Avalability
    candidates = candidates[candidates["in_stock"] == True]

    # 7️⃣ Hybrid Score
    sales_max = candidates["sales_count"].max()
    sales_norm = candidates["sales_count"] / sales_max if sales_max > 0 else 0

    candidates["hybrid_score"] = (
        0.5 * candidates["high_rating_prob"] +
        0.3 * sales_norm +
        0.2 * candidates["query_similarity"]
    )

    candidates = candidates.sort_values("hybrid_score", ascending=False)

    return candidates[
        [
            "product_name",
            "category",
            "brand",
            "price",
            "rating",
            "high_rating_prob",
            "sales_count",
            "hybrid_score",
            "image_url",
             "in_stock"
        ]
    ].head(top_n)



Device set to use cpu


In [50]:
df_clean["image_url"] = (
    df_clean["image_url"]
    .fillna("")          # نعالج الـ NaN
    .astype(str)         # نضمن إنه string
    .str.strip()         # نشيل مسافات
)

In [54]:
# Format Recommendations as Product Cards
def format_products_as_cards(recommendation_df):
    cards = []
    for _, row in recommendation_df.iterrows():
        card = {
            "type": "product_card",
            "image": {
                "url": row["image_url"],          # الصورة مرتبطة بالبرودكت مباشرة
                "alt": row["product_name"]
            },
            "product_info": {
                "name": row["product_name"],
                "brand": row["brand"],
                "price": f'{row["price"]} EGP',
                "rating": {
                    "value": round(row["rating"], 1),  # عدد النجوم 1-5
                    "max": 5
                },
                "availability": "Available" if row["in_stock"] else "Out of Stock"
                #",product_url": row["real_product_url"]  #   وده اسم  الكولوم الخاص برابط صفحه البرودكت
            }
        }
        cards.append(card)
    return cards


In [59]:
# ============================================
#  Welcome Messages (Introductory)
# ============================================
import random


welcome_messages = [
    "👋 Welcome! I'm your personal shopping assistant. Please type what you're looking for or describe your needs, and I'll suggest the best products for you. Whether it's gifts, essentials, or something special, I'm here to guide you!",
    "Hello and welcome! 🛍️ Tell me exactly what you need or the type of product you're interested in, and I'll provide top recommendations. Start typing your request now to see tailored options!",
    "Hi there! 🌟 To get started, just type your request or describe the product you want. I'll find the most suitable products for you quickly and easily. Let's make your shopping fun and simple!",
    "Welcome! 🎯 Share what you are looking for or describe your needs, and I'll recommend products that match your preferences. Start typing to explore the best options now!",
    "Hey! 🌟 Whether you have a specific product in mind or just a general idea, type it here and I’ll guide you to the best recommendations available."

]

def show_welcome_message():
    message = random.choice(welcome_messages)
    print("\n" + message + "\n")

# ============================================
#  Post-Recommendation Messages
# ============================================
post_recommendation_messages = [
    "Here are the products I found for you! 💡 Click on any product card to visit its page and see more details."
    ,"You can also ask me for recommendations from a specific brand or within a certain price range if you want.",
    "These recommendations are tailored for you! 🛒 Simply click any product to explore its page. "
     ,"If you want, I can also guide you to options based on your favorite brands or budget.",
    "I hope you find these products interesting! ⭐ Click on a product to view full details and make your choice."
     ,"You can also ask me for similar items or products from a specific brand.",
    "Here are your top picks! 🎯 Each product card is clickable so you can check the product page directly."
    ,"I hope you find these recommendations useful!"
]

def show_post_recommendation_message():
    message = random.choice(post_recommendation_messages)
    print("\n" + message + "\n")


In [61]:
import random
import signal

# ================================
#  Timeout input function
# ================================
def input_with_timeout(prompt, timeout=60):
    def handler(signum, frame):
        raise TimeoutError

    signal.signal(signal.SIGALRM, handler)
    signal.alarm(timeout)
    try:
        user_input = input(prompt)
        signal.alarm(0)
        return user_input
    except TimeoutError:
        print("\nNo input detected for 1 minute. Exiting the chatbot. ")
        exit()

# ================================
# Greeting & farewell words
# ================================
greetings = ["hello", "hi", "hey", "greetings", "good morning", "good evening", "hey there","Hi","Hay","hay","help me ","Help me ","can you help me",]
farewell_triggers = ["Okay", "Thank you ", "bye","ok"," baye","thx","Thx","thanks for help","Thanks for your help","good job "]

# ================================
# Greeting responses
# ================================
greeting_responses = [
    "Hey! How can I assist you today?",
    "Hello!  Ready to help you find the perfect product.",
    "Hi there! Please tell me what you need.",
    "Greetings!  What can I help you find today?"
]

# ================================
# Farewell responses
# ================================
farewell_responses = [
    "Thank you for using the system! ",
    "Goodbye! Hope to see you again soon.",
    "Exit acknowledged. Take care! ",
    "Bye! Feel free to come back anytime for more recommendations."
]

# ================================
# Manual Test / Interaction Loop
# ================================
def manual_test_interaction():
    print("===== Welcome to the Recommendation Chatbot =====\n")

    # رسالة ترحيبية من قائمة الترحيبات العامة
    print(random.choice(welcome_messages))
    print("\nType 'quit' to exit anytime.\n")

    while True:
        try:
            user_input = input_with_timeout("Your request: ", timeout=60).strip()
        except EOFError:
            print("\nInput closed. Exiting chatbot.")
            break

        # الرد على التحيات
        if any(greet in user_input.lower() for greet in greetings):
            print("Bot:", random.choice(greeting_responses))
            continue

        # الرد على الخروج
        if user_input.lower() in farewell_triggers:
            print("Bot:", random.choice(farewell_responses))
            break

        # فحص الانبوت
        is_valid, message = check_user_input(user_input)
        if not is_valid:
            print("\n" + message + "\n")
            continue

        # فحص لو الانبوت مش مفيد (لا كاتيجوري، لا براند، لا برودكت)
        detected_categories = interpret_user_request(user_input)
        extracted_brand = extract_brand(user_input)
        extracted_price = extract_price_range(user_input)

        if not detected_categories and not extracted_brand and not extracted_price:
            print("Bot: Sorry, I couldn't find any useful information in your request. "
                  "Please describe your product needs more clearly.")
            continue

        # استدعاء الموديل للريكومنديشن
        recommendations = hybrid_recommendation(user_input, top_n=5)

        if recommendations.empty:
            print("\nBot: Sorry, no matching products found. Try describing your needs differently!\n")
            continue

        # تنسيق النتائج كـ Cards
        cards = format_products_as_cards(recommendations)

        print("\n===== Top Recommendations =====\n")
        for idx, card in enumerate(cards, start=1):
            print(f"--- Product #{idx} ---")
            print(f"Image URL      : {card['image']['url']}")
            print(f"Name           : {card['product_info']['name']}")
            print(f"Brand          : {card['product_info']['brand']}")
            print(f"Price          : {card['product_info']['price']}")
            print(f"Rating         : {card['product_info']['rating']['value']} / {card['product_info']['rating']['max']}")
            print(f"Availability   : {card['product_info']['availability']}")
            # لو عندك real_product_url فعليًا:
            # print(f"Product Page   : {card['product_info']['product_url']}")

        # رسالة بعد الريكومنديشن
        print("Bot:", random.choice(post_recommendation_messages))
        print("="*60 + "\n")

# تشغيل التفاعل اليدوي
manual_test_interaction()


===== Welcome to the Recommendation Chatbot =====

Hello and welcome! 🛍️ Tell me exactly what you need or the type of product you're interested in, and I'll provide top recommendations. Start typing your request now to see tailored options!

Type 'quit' to exit anytime.

Your request: hih
Bot: Hey! How can I assist you today?
Your request: adce

Your input is too short or unclear. Please provide more details.

Your request: hihi
Bot: Greetings!  What can I help you find today?
Your request: wergebtbryehyn

Your input is too short or unclear. Please provide more details.

Your request: ىتىتىةمتن

Please write your request in English so I can understand it.

Your request: can you help me
Bot: Sorry, I couldn't find any useful information in your request. Please describe your product needs more clearly.
Your request: hello
Bot: Hi there! Please tell me what you need.
Your request: ok 
Bot: Bye! Feel free to come back anytime for more recommendations.
